# Homologación Municipios

## 1. LIBRERÍAS Y RUTAS

In [15]:
import pandas as pd
import numpy as np
 
# ── Rutas de los datasets en Kaggle ──────────────────────────────────────────
RUTA_DEFUNCIONES = "/kaggle/input/datasets/nicolasacostaa/defunciones-consolidadas/defunciones_consolidadas.parquet"
RUTA_DIVIPOLA    = "/kaggle/input/datasets/nicolasacostaa/municipios-divipola/DIVIPOLA_Municipios.xlsx - Municipios.csv"
RUTA_LISTA_105   = "/kaggle/input/datasets/nicolasacostaa/lista-105-defunciones-colombia/Lista_105_Colombia_CIE9-y-CIE10.xls"

## 2. FUNCIONES

### 2.1. CARGAR DEFUNCIONES

In [16]:
def cargar_defunciones(ruta: str) -> pd.DataFrame:
    """
    Carga el archivo parquet de defunciones consolidadas.
 
    Parameters
    ----------
    ruta : str
        Ruta al archivo .parquet dentro del entorno de Kaggle.
 
    Returns
    -------
    pd.DataFrame
        DataFrame con las defunciones consolidadas.
    """
    print(f"[INFO] Cargando defunciones desde: {ruta}")
    df = pd.read_parquet(ruta)
    print(f"  → Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
    return df

### 2.2.Cargar Divipola

In [17]:
 def cargar_divipola(ruta: str) -> pd.DataFrame:
    """
    Carga el archivo CSV de municipios DIVIPOLA.
 
    Parameters
    ----------
    ruta : str
        Ruta al archivo CSV dentro del entorno de Kaggle.
 
    Returns
    -------
    pd.DataFrame
        DataFrame con la tabla de municipios.
    """
    print(f"[INFO] Cargando DIVIPOLA desde: {ruta}")
    df = pd.read_csv(ruta, dtype={"cod_muni": str, "doc_dep": str})
    print(f"  → Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
    return df

### 2.3. Cargar Lista 105

In [18]:
def cargar_lista_105(ruta: str) -> pd.DataFrame:
    """
    Carga y limpia el archivo XLS de la Lista Colombia 105 para
    tabulación de mortalidad (CIE-10 y CIE-9).
 
    Proceso de limpieza:
      - Detecta automáticamente la fila de encabezado real (fila índice 3).
      - Renombra columnas al estándar requerido.
      - Elimina filas vacías, de título y de pie de página.
      - Normaliza 'No. Lista' a string de 3 dígitos con cero a la
        izquierda para permitir el cruce con defunciones.
        Ejemplo: '01' → '001'  |  '105' → '105'
 
    Parameters
    ----------
    ruta : str
        Ruta al archivo .xls dentro del entorno de Kaggle.
 
    Returns
    -------
    pd.DataFrame
        df_lista_105 con columnas:
          ['No. Lista', 'Causa', 'Códigos CIE-10', 'Códigos CIE-9']
    """
    print(f"[INFO] Cargando Lista 105 desde: {ruta}")
 
    # El encabezado real está en la fila índice 3 (base 0)
    df = pd.read_excel(ruta, engine="xlrd", header=3)
    df.columns = ["No. Lista", "Causa", "Códigos CIE-10", "Códigos CIE-9"]
 
    # Eliminar filas completamente vacías
    df = df.dropna(how="all").reset_index(drop=True)
 
    # Conservar solo filas donde 'No. Lista' sea un código numérico puro
    # (excluye filas de título, encabezado duplicado y pie de página)
    mascara_valida = df["No. Lista"].astype(str).str.strip().str.match(r"^\d+$")
    df = df[mascara_valida].copy().reset_index(drop=True)
 
    # Normalizar 'No. Lista' a 3 dígitos con cero a la izquierda
    # para que sea comparable con 'CÓDIGO CAUSA BÁSICA LISTA 105' de defunciones
    # Ejemplo: '01' → '001'  |  '105' → '105'
    df["No. Lista"] = df["No. Lista"].astype(str).str.strip().str.zfill(3)
 
    # Limpiar espacios residuales en campos de texto
    for col in ["Causa", "Códigos CIE-10", "Códigos CIE-9"]:
        df[col] = df[col].astype(str).str.strip()
 
    print(f"  → Registros cargados : {len(df)}")
    print(f"  → 'No. Lista' muestra: {df['No. Lista'].head(5).tolist()}")
    return df

### 2.4. Crear Codigo Municipio

In [19]:
def crear_codigo_municipio(df: pd.DataFrame,
                            col_dep: str  = "CÓDIGO DEPARTAMENTO",
                            col_muni: str = "CÓDIGO MUNICIPIO",
                            col_nueva: str = "cod_muni_def") -> pd.DataFrame:
    """
    Crea un campo nuevo en el DataFrame de defunciones que es la
    concatenación de código departamento + código municipio, ambos
    alineados con el formato de DIVIPOLA (5 dígitos, cero a la izquierda).
 
    Ejemplo: dep='05', muni='001'  →  cod_muni_def='05001'
 
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de defunciones.
    col_dep : str
        Nombre de la columna código departamento.
    col_muni : str
        Nombre de la columna código municipio.
    col_nueva : str
        Nombre del campo resultante.
 
    Returns
    -------
    pd.DataFrame
        DataFrame con el campo nuevo añadido.
    """
    print(f"[INFO] Creando campo '{col_nueva}' = {col_dep} + {col_muni}")
    df = df.copy()
 
    # Asegurar formato string y padding correcto (dep=2 dígitos, muni=3 dígitos)
    dep  = df[col_dep].astype(str).str.strip().str.zfill(2)
    muni = df[col_muni].astype(str).str.strip().str.zfill(3)
 
    df[col_nueva] = dep + muni
 
    # Verificación rápida
    n_nulos = df[col_nueva].isna().sum()
    print(f"  → Valores nulos en '{col_nueva}': {n_nulos:,}")
    print(f"  → Muestra: {df[col_nueva].unique()[:5].tolist()}")
    return df

### 2.5. Normalizar Código Municipio

In [20]:
def normalizar_cod_muni_divipola(df: pd.DataFrame,
                                  col: str = "cod_muni") -> pd.DataFrame:
    """
    Normaliza la columna cod_muni de DIVIPOLA a string de 5 dígitos
    con cero a la izquierda, para que sea comparable con cod_muni_def.
 
    Ejemplo: 91263 (int)  →  '91263' (str de 5 dígitos)
 
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de DIVIPOLA.
    col : str
        Nombre de la columna a normalizar.
 
    Returns
    -------
    pd.DataFrame
        DataFrame con la columna normalizada.
    """
    print(f"[INFO] Normalizando '{col}' en DIVIPOLA a string de 5 dígitos")
    df = df.copy()
    df[col] = df[col].astype(str).str.strip().str.zfill(5)
    print(f"  → Muestra: {df[col].unique()[:5].tolist()}")
    return df

### 2.6. Hacer el Join con Divipola

In [21]:
def cruzar_con_divipola(defunciones: pd.DataFrame,
                         divipola: pd.DataFrame,
                         col_izq: str = "cod_muni_def",
                         col_der: str = "cod_muni",
                         tipo: str    = "left") -> pd.DataFrame:
    """
    Realiza el cruce entre el DataFrame de defunciones y DIVIPOLA
    usando las claves de municipio normalizadas.
 
    Parameters
    ----------
    defunciones : pd.DataFrame
        DataFrame de defunciones con el campo cod_muni_def ya creado.
    divipola : pd.DataFrame
        DataFrame de DIVIPOLA con cod_muni normalizado.
    col_izq : str
        Llave de cruce en defunciones.
    col_der : str
        Llave de cruce en DIVIPOLA.
    tipo : str
        Tipo de join: 'left' (default), 'inner', 'right', 'outer'.
 
    Returns
    -------
    pd.DataFrame
        DataFrame resultante del cruce.
    """
    print(f"[INFO] Cruzando defunciones ({col_izq}) ↔ DIVIPOLA ({col_der}) | tipo={tipo}")
 
    # Columnas de DIVIPOLA que se traen (excluir la llave para evitar duplicado)
    cols_divipola = [c for c in divipola.columns if c != col_der]
 
    df_merged = defunciones.merge(
        divipola[[col_der] + cols_divipola],
        left_on  = col_izq,
        right_on = col_der,
        how      = tipo
    )
 
    # Diagnóstico del cruce
    n_total     = len(defunciones)
    n_cruzados  = df_merged[col_der].notna().sum()
    n_sin_cruce = df_merged[col_der].isna().sum()
    pct         = n_cruzados / n_total * 100
 
    print(f"  → Registros originales : {n_total:>12,}")
    print(f"  → Con cruce exitoso    : {n_cruzados:>12,}  ({pct:.2f}%)")
    print(f"  → Sin cruce (NaN)      : {n_sin_cruce:>12,}")
    print(f"  → Shape final         : {df_merged.shape}")
 
    return df_merged
 

### 2.7. Hacer el Join con Divipola

In [22]:
def cruzar_con_lista_105(df: pd.DataFrame,
                          lista_105: pd.DataFrame,
                          col_izq: str = "CÓDIGO CAUSA BÁSICA LISTA 105",
                          col_der: str = "No. Lista",
                          tipo: str    = "left") -> pd.DataFrame:
    """
    Cruza el DataFrame de defunciones con la Lista 105 para obtener
    la descripción de la causa de muerte (Causa, CIE-10, CIE-9).
 
    La normalización de 'CÓDIGO CAUSA BÁSICA LISTA 105' a 3 dígitos
    con cero a la izquierda se realiza automáticamente dentro de esta
    función, por lo que es compatible con cualquier formato de entrada.
 
    Ejemplo de normalización: '81' → '081'  |  '1' → '001'
 
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de defunciones (puede ser el resultado del cruce
        previo con DIVIPOLA).
    lista_105 : pd.DataFrame
        df_lista_105 generado por cargar_lista_105().
    col_izq : str
        Columna de causa en defunciones.
    col_der : str
        Columna 'No. Lista' en lista_105.
    tipo : str
        Tipo de join.
 
    Returns
    -------
    pd.DataFrame
        DataFrame enriquecido con Causa, Códigos CIE-10 y CIE-9.
    """
    print(f"[INFO] Cruzando con Lista 105 ({col_izq}) ↔ ({col_der}) | tipo={tipo}")
    df = df.copy()
 
    # Normalizar clave en defunciones a 3 dígitos con cero a la izquierda
    df[col_izq] = df[col_izq].astype(str).str.strip().str.zfill(3)
 
    df_merged = df.merge(
        lista_105[[col_der, "Causa", "Códigos CIE-10", "Códigos CIE-9"]],
        left_on=col_izq, right_on=col_der, how=tipo
    )
 
    n_total     = len(df)
    n_cruzados  = df_merged[col_der].notna().sum()
    n_sin_cruce = df_merged[col_der].isna().sum()
 
    print(f"  → Registros originales : {n_total:>12,}")
    print(f"  → Con cruce exitoso    : {n_cruzados:>12,}  ({n_cruzados/n_total*100:.2f}%)")
    print(f"  → Sin cruce (NaN)      : {n_sin_cruce:>12,}")
    print(f"  → Shape final          : {df_merged.shape}")
    return df_merged

### 2.8. Diagnosticar no cruzan Municipio

In [23]:
def diagnosticar_sin_cruce_municipio(df_merged: pd.DataFrame,
                                      col_key: str   = "cod_muni_def",
                                      col_check: str = "municipio") -> pd.DataFrame:
    """
    Identifica los códigos de municipio de defunciones que NO encontraron
    correspondencia en DIVIPOLA, con conteo de registros y fallecidos.
 
    Parameters
    ----------
    df_merged : pd.DataFrame
        DataFrame resultado del cruce con DIVIPOLA (left join).
    col_key : str
        Columna de clave de municipio en defunciones.
    col_check : str
        Columna de DIVIPOLA para detectar filas sin cruce (NaN = sin match).
 
    Returns
    -------
    pd.DataFrame
        Resumen de códigos sin cruce, ordenado por registros desc.
    """
    sin_cruce = (
        df_merged[df_merged[col_check].isna()]
        [[col_key, "Número de Fallecidos"]]
        .groupby(col_key, as_index=False)
        .agg(registros=("Número de Fallecidos", "count"),
             fallecidos=("Número de Fallecidos", "sum"))
        .sort_values("registros", ascending=False)
    )
    print(f"[INFO] Municipios sin cruce: {len(sin_cruce)} código(s) distinto(s)")
    return sin_cruce

### 2.9. Diagnosticar no cruzan Lista 105

In [24]:
def diagnosticar_sin_cruce_lista_105(df_merged: pd.DataFrame,
                                      col_key: str   = "CÓDIGO CAUSA BÁSICA LISTA 105",
                                      col_check: str = "Causa") -> pd.DataFrame:
    """
    Identifica los códigos de Lista 105 de defunciones que NO encontraron
    correspondencia en la tabla de causas, con conteo de registros y fallecidos.
 
    Parameters
    ----------
    df_merged : pd.DataFrame
        DataFrame resultado del cruce con Lista 105 (left join).
    col_key : str
        Columna de clave de causa en defunciones.
    col_check : str
        Columna de Lista 105 para detectar filas sin cruce (NaN = sin match).
 
    Returns
    -------
    pd.DataFrame
        Resumen de códigos de causa sin cruce, ordenado por registros desc.
    """
    sin_cruce = (
        df_merged[df_merged[col_check].isna()]
        [[col_key, "Número de Fallecidos"]]
        .groupby(col_key, as_index=False)
        .agg(registros=("Número de Fallecidos", "count"),
             fallecidos=("Número de Fallecidos", "sum"))
        .sort_values("registros", ascending=False)
    )
    print(f"[INFO] Códigos Lista 105 sin cruce: {len(sin_cruce)} código(s) distinto(s)")
    return sin_cruce
 

### 2.10. Limpieza - Excluir Registros sin Cruce

In [25]:
def excluir_sin_cruce(df: pd.DataFrame,
                       col_check: str,
                       nombre_salida: str = "df_limpio") -> pd.DataFrame:
    """
    Filtra el DataFrame eliminando las filas cuyo cruce no fue exitoso,
    es decir, donde la columna de verificación es NaN.
 
    Función genérica: sirve tanto para el cruce de municipios como para
    el cruce de causas Lista 105 (o cualquier otro cruce futuro).
 
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame resultado de algún cruce (left join).
    col_check : str
        Columna que indica si el cruce fue exitoso (NaN = sin cruce).
    nombre_salida : str
        Nombre descriptivo del DataFrame resultante (solo para el log).
 
    Returns
    -------
    pd.DataFrame
        DataFrame limpio sin filas con cruce fallido.
    """
    n_antes = len(df)
    df_limpio = df[df[col_check].notna()].copy().reset_index(drop=True)
    n_despues = len(df_limpio)
    n_excluidos = n_antes - n_despues
 
    print(f"[INFO] Limpieza → {nombre_salida}")
    print(f"  → Filas antes    : {n_antes:>12,}")
    print(f"  → Filas excluidas: {n_excluidos:>12,}  ({n_excluidos/n_antes*100:.2f}%)")
    print(f"  → Filas después  : {n_despues:>12,}")
    return df_limpio


### 2.11. Exportar a formato parquet

In [26]:
def exportar_parquet(df: pd.DataFrame,
                     ruta_salida: str,
                     compresion: str = "snappy") -> None:
    """
    Exporta cualquier DataFrame al formato Parquet.
 
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a exportar.
    ruta_salida : str
        Ruta completa de destino, incluyendo nombre y extensión .parquet.
        En Kaggle usa '/kaggle/working/nombre_archivo.parquet'.
    compresion : str
        Algoritmo de compresión: 'snappy' (default), 'gzip' o 'brotli'.
 
    Returns
    -------
    None
    """
    print(f"[INFO] Exportando a Parquet: {ruta_salida}")
    df.to_parquet(ruta_salida, index=False, compression=compresion)
    print(f"  → Guardado  | Filas: {len(df):,} | Columnas: {df.shape[1]}")

### 2.12. PIPELINE PRINCIPAL

In [27]:
def main():
 
    # ── 7.1 Carga ─────────────────────────────────────────────────────────────
    defunciones  = cargar_defunciones(RUTA_DEFUNCIONES)
    divipola     = cargar_divipola(RUTA_DIVIPOLA)
    df_lista_105 = cargar_lista_105(RUTA_LISTA_105)
 
    print("\n[INFO] Muestra df_lista_105:")
    display(df_lista_105.head(5))
 
    # ── 7.2 Preparación de claves ─────────────────────────────────────────────
    defunciones = crear_codigo_municipio(defunciones)
    divipola    = normalizar_cod_muni_divipola(divipola)
 
    # ── 7.3 Cruce con municipios DIVIPOLA ─────────────────────────────────────
    df_con_muni = cruzar_con_divipola(defunciones, divipola)
 
    print("\n─── Diagnóstico: municipios sin cruce ───")
    sin_cruce_muni = diagnosticar_sin_cruce_municipio(df_con_muni)
    if not sin_cruce_muni.empty:
        display(sin_cruce_muni)
 
    # ── 7.4 Limpieza 1: eliminar registros sin municipio ──────────────────────
    df_limpio_consolidado_defunciones = excluir_sin_cruce(
        df_con_muni,
        col_check="municipio",
        nombre_salida="df_limpio_consolidado_defunciones"
    )
    print("\n[INFO] Muestra df_limpio_consolidado_defunciones:")
    display(df_limpio_consolidado_defunciones.head(5))
 
    # ── 7.5 Cruce con Lista 105 (causa de muerte) ─────────────────────────────
    df_con_causa = cruzar_con_lista_105(df_limpio_consolidado_defunciones, df_lista_105)
 
    print("\n─── Diagnóstico: causas Lista 105 sin cruce ───")
    sin_cruce_causa = diagnosticar_sin_cruce_lista_105(df_con_causa)
    if not sin_cruce_causa.empty:
        display(sin_cruce_causa)
    else:
        print("[OK] Todos los códigos de causa cruzaron exitosamente.")
 
    # ── 7.6 Limpieza 2: eliminar registros sin causa ──────────────────────────
    df_limpio_consolidado_defunciones_causa_ubicacion = excluir_sin_cruce(
        df_con_causa,
        col_check="Causa",
        nombre_salida="df_limpio_consolidado_defunciones_causa_ubicacion"
    )
    print("\n[INFO] Muestra df_limpio_consolidado_defunciones_causa_ubicacion:")
    display(df_limpio_consolidado_defunciones_causa_ubicacion.head(5))
 
    # ── 7.7 Exportación a Parquet ─────────────────────────────────────────────
    exportar_parquet(
        df_limpio_consolidado_defunciones,
        "/kaggle/working/df_limpio_consolidado_defunciones.parquet"
    )
    exportar_parquet(
        df_limpio_consolidado_defunciones_causa_ubicacion,
        "/kaggle/working/df_limpio_consolidado_defunciones_causa_ubicacion.parquet"
    )
 
    return (
        df_lista_105,
        df_limpio_consolidado_defunciones,
        df_limpio_consolidado_defunciones_causa_ubicacion
    )

### 3. EJECUCIÓN

In [28]:
(
    df_lista_105,
    df_limpio_consolidado_defunciones,
    df_limpio_consolidado_defunciones_causa_ubicacion
) = main()

[INFO] Cargando defunciones desde: /kaggle/input/datasets/nicolasacostaa/defunciones-consolidadas/defunciones_consolidadas.parquet
  → Filas: 8,701,084  |  Columnas: 15
[INFO] Cargando DIVIPOLA desde: /kaggle/input/datasets/nicolasacostaa/municipios-divipola/DIVIPOLA_Municipios.xlsx - Municipios.csv
  → Filas: 1,122  |  Columnas: 4
[INFO] Cargando Lista 105 desde: /kaggle/input/datasets/nicolasacostaa/lista-105-defunciones-colombia/Lista_105_Colombia_CIE9-y-CIE10.xls
  → Registros cargados : 105
  → 'No. Lista' muestra: ['001', '002', '003', '004', '005']

[INFO] Muestra df_lista_105:


,No. Lista,Causa,Códigos CIE-10,Códigos CIE-9
0,001,Enfermedades infecciosas intestinales,A00-A09,"001-009, 136.5"
1,002,Tuberculosis y secuelas,"A15-A19, B90","010-018, 137"
2,003,Ciertas enfermedades transmitidas por vectores...,"A20, A44, A75-A79, A82-A84, A85.2, A90-A98, B5...","020, 060-066, 071, 078.6-078.8, 080-088"
3,004,Ciertas enfermedades inmunoprevenibles,"A33-A37, A80, B05-B06, B26, B91","032-033, 037, 045, 055-056, 072, 138, 771,3"
4,005,"Septicemia, excepto neonatal",A40-A41,038


[INFO] Creando campo 'cod_muni_def' = CÓDIGO DEPARTAMENTO + CÓDIGO MUNICIPIO
  → Valores nulos en 'cod_muni_def': 0
  → Muestra: ['05001', '05002', '05004', '05021', '05030']
[INFO] Normalizando 'cod_muni' en DIVIPOLA a string de 5 dígitos
  → Muestra: ['91263', '91405', '91407', '91430', '91001']
[INFO] Cruzando defunciones (cod_muni_def) ↔ DIVIPOLA (cod_muni) | tipo=left
  → Registros originales :    8,701,084
  → Con cruce exitoso    :    8,692,830  (99.91%)
  → Sin cruce (NaN)      :        8,254
  → Shape final         : (8701084, 20)

─── Diagnóstico: municipios sin cruce ───
[INFO] Municipios sin cruce: 51 código(s) distinto(s)


,cod_muni_def,registros,fallecidos
28,83001,1699,1777
6,11279,1224,1232
7,11769,1034,1037
36,83592,543,575
31,83247,509,529
8,11848,505,508
4,11102,478,481
0,05387,313,316
37,83753,243,256
9,11850,242,246


[INFO] Limpieza → df_limpio_consolidado_defunciones
  → Filas antes    :    8,701,084
  → Filas excluidas:        8,254  (0.09%)
  → Filas después  :    8,692,830

[INFO] Muestra df_limpio_consolidado_defunciones:


,CÓDIGO DEPARTAMENTO,CÓDIGO MUNICIPIO,ÁREA DEFUNCIÓN,AÑO,MES,SEXO,AGRUPACIÓN EDADES 1,ESTADO CIVIL,CÓDIGO DEPARTAMENTO RESIDENCIA,CÓDIGO MUNICIPIO RESIDENCIA,CÓDIGO SITIO DEFUNCIÓN,CÓDIGO CAUSA BÁSICA DEFUNCIÓN,CÓDIGO CERTIFICACIÓN MÉDICA,CÓDIGO CAUSA BÁSICA LISTA 105,Número de Fallecidos,cod_muni_def,cod_muni,doc_dep,departamento,municipio
0,05,001,1,1979,01,1,01,1,05,001,1,7651,1,081,7,05001,05001,05,ANTIOQUIA,MEDELLÍN
1,05,001,1,1979,01,1,01,1,05,001,1,7651,2,081,1,05001,05001,05,ANTIOQUIA,MEDELLÍN
2,05,001,1,1979,01,1,01,1,05,001,1,7679,1,080,2,05001,05001,05,ANTIOQUIA,MEDELLÍN
3,05,001,1,1979,01,1,01,1,05,001,1,7684,1,082,1,05001,05001,05,ANTIOQUIA,MEDELLÍN
4,05,001,1,1979,01,1,01,1,05,001,1,7684,2,082,1,05001,05001,05,ANTIOQUIA,MEDELLÍN


[INFO] Cruzando con Lista 105 (CÓDIGO CAUSA BÁSICA LISTA 105) ↔ (No. Lista) | tipo=left
  → Registros originales :    8,692,830
  → Con cruce exitoso    :    8,692,830  (100.00%)
  → Sin cruce (NaN)      :            0
  → Shape final          : (8692830, 24)

─── Diagnóstico: causas Lista 105 sin cruce ───
[INFO] Códigos Lista 105 sin cruce: 0 código(s) distinto(s)
[OK] Todos los códigos de causa cruzaron exitosamente.
[INFO] Limpieza → df_limpio_consolidado_defunciones_causa_ubicacion
  → Filas antes    :    8,692,830
  → Filas excluidas:            0  (0.00%)
  → Filas después  :    8,692,830

[INFO] Muestra df_limpio_consolidado_defunciones_causa_ubicacion:


,CÓDIGO DEPARTAMENTO,CÓDIGO MUNICIPIO,ÁREA DEFUNCIÓN,AÑO,MES,SEXO,AGRUPACIÓN EDADES 1,ESTADO CIVIL,CÓDIGO DEPARTAMENTO RESIDENCIA,CÓDIGO MUNICIPIO RESIDENCIA,...,Número de Fallecidos,cod_muni_def,cod_muni,doc_dep,departamento,municipio,No. Lista,Causa,Códigos CIE-10,Códigos CIE-9
0,05,001,1,1979,01,1,01,1,05,001,...,7,05001,05001,05,ANTIOQUIA,MEDELLÍN,081,"Retardo del crecimiento fetal, desnutrición fe...","P05, P07",764-765
1,05,001,1,1979,01,1,01,1,05,001,...,1,05001,05001,05,ANTIOQUIA,MEDELLÍN,081,"Retardo del crecimiento fetal, desnutrición fe...","P05, P07",764-765
2,05,001,1,1979,01,1,01,1,05,001,...,2,05001,05001,05,ANTIOQUIA,MEDELLÍN,080,Feto y recién nacido afectados por complicacio...,"P01-P03, P10-P15","761-763.4, 763.6-763.9, 767"
3,05,001,1,1979,01,1,01,1,05,001,...,1,05001,05001,05,ANTIOQUIA,MEDELLÍN,082,Trastornos respiratorios específicos del perío...,P20-P28,768-770
4,05,001,1,1979,01,1,01,1,05,001,...,1,05001,05001,05,ANTIOQUIA,MEDELLÍN,082,Trastornos respiratorios específicos del perío...,P20-P28,768-770


[INFO] Exportando a Parquet: /kaggle/working/df_limpio_consolidado_defunciones.parquet
  → Guardado  | Filas: 8,692,830 | Columnas: 20
[INFO] Exportando a Parquet: /kaggle/working/df_limpio_consolidado_defunciones_causa_ubicacion.parquet
  → Guardado  | Filas: 8,692,830 | Columnas: 24
